In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

products = spark.table("novacart_dev.silver.slv_products")

In [0]:
print(f"  - Products: {products.count():,} rows")

In [0]:
# Dimension: Products (clean, no nulls)
dim_products = products \
    .filter(F.col("product_id").isNotNull()) \
    .select(
        F.col("product_id").alias("product_key"),
        "product_name",
        "category",
        "price",
        "currency",
        F.col("country_code").alias("product_origin_country"),
        "load_timestamp"
    ) \
    .distinct()

print(f"dim_products: {dim_products.count():,} products")
display(dim_products.limit(5))

In [0]:
# Write dim_products to Gold layer
dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("novacart_dev.gold.dim_products")

print("✓ dim_products written to novacart_dev.gold.dim_products")